# Build the evidence knowledge graph

Run each protected stage deliberately. Every write stage below documents its inputs, outputs, database effect, rerun behavior, and invalidation rules; the following cell is read-only inspection.

In [1]:
from pathlib import Path
from potency_kg.alias_discovery import DEFAULT_ENABLE_LLM_ALIAS_DISCOVERY
from potency_kg.workflow_stages import (
    classify_jev_stage, discover_aliases_deterministic_stage, discover_aliases_llm_stage,
    embed_stage, export_stage, extract_stage, import_stage, inspect_aliases_stage,
    inspect_alias_evidence_stage, inspect_stage, load_alias_seeds_stage, materialize_stage,
    preflight_stage, refresh_graph_stage
)
working_directory = Path.cwd().resolve()
package_root = working_directory.parent if working_directory.name == 'notebooks' else working_directory
config_path = package_root / 'configs/casgevy-potency.json'
output_root = package_root / 'results/kg'
enable_llm_alias_discovery = DEFAULT_ENABLE_LLM_ALIAS_DISCOVERY


## Preflight
Inputs: supplied descriptor and corpus. Outputs: validated hashes and counts. Database effect: none. Rerun behavior: safe and read-only. Invalidation: changed supplied inputs require a new import.

In [2]:
preflight_report = preflight_stage(config_path)
preflight_report


CorpusPreflightReport(descriptor_sha256='714d54dee46bf3843de0677ef224352509c9ce8faff6588d8a30893887646c5e', manifest_sha256='126a464c5c6df12710b9ac3a287d2b5947240196de177e4a7cd69fed2ece7d27', records_sha256='34f547dce564414c2100bcc4d48f287fffae8b0b4bfc2ddeb08a72198f1e5f60', raw_sha256={'casgevy-125785-analytical-method-review': 'aff6f3891ba1da965620d69cb1400a9893f22f33673c820144bfcb1ceda72255', 'casgevy-125785-approval-letter': '489de36be50b17defb8c964df772a5c1bbe9be9bd95938d4fb6dfa5c906a2b7a', 'casgevy-125785-bioresearch-monitoring-final-review-memo-casgevy': '1edf0e5e0f9da3b57fc87d72cdf727bacb1d0a086cf30e048e8391b8688eedba', 'casgevy-125785-cber-bioinformatics-bla-review-memo-casgevy': '8bb2af475a0f7b0def9f98dc39224ca6a3dd0a4201bc9ac0bc480e7816c6e13a', 'casgevy-125785-cber-dmpq-cmc-facility-bla-review-memo-casgevy': '6e2ac65c659c16138a0d62fe4f6c4b10345c4d0517bb05d373c4b4bf2759116a', 'casgevy-125785-clinical-pharmacology-bla-review-casgevy': 'fbedfe2ab305a2fa77d8305d0162674aa5de9c397a

In [3]:
preflight_inspection = {'document_count': getattr(preflight_report, 'document_count', None)}
preflight_inspection


{'document_count': 24}

## Structural import
Inputs: preflight-valid descriptor. Outputs: source and passage report. Database effect: writes structural rows. Rerun behavior: idempotent. Invalidation: a new source version requires later alias and semantic stages.

In [4]:
import_report = import_stage(config_path)
import_report


ImportReport(source_count=24, passage_outcome_count=791, imported_passages=791, invalid_passage_outcomes=0, new_source_versions=24, unchanged_source_versions=0, structural_nodes=815, structural_edges=791, per_source_counts={'casgevy-125785-analytical-method-review': 18, 'casgevy-125785-approval-letter': 7, 'casgevy-125785-bioresearch-monitoring-final-review-memo-casgevy': 2, 'casgevy-125785-cber-bioinformatics-bla-review-memo-casgevy': 53, 'casgevy-125785-cber-dmpq-cmc-facility-bla-review-memo-casgevy': 93, 'casgevy-125785-clinical-pharmacology-bla-review-casgevy': 64, 'casgevy-125785-cmc-review': 140, 'casgevy-125785-employee-officer-list-casgevy': 2, 'casgevy-125785-labeling-review-casgevy': 9, 'casgevy-125785-late-cycle-summary': 9, 'casgevy-125785-mid-cycle-summary': 10, 'casgevy-125785-pharmacology-toxicology-review-casgevy': 74, 'casgevy-125785-pharmacovigilance-plan-review-memo-casgevy': 40, 'casgevy-125785-statistical-review-casgevy': 44, 'casgevy-125787-scd-approval-letter': 1

In [5]:
import_inspection = inspect_stage(config_path, output_root=output_root)
import_inspection


{'scope': Scope(tenant_id='public-fda', project_id='casgevy-potency'),
 'catalog': AliasCatalogStatus(scope=Scope(tenant_id='public-fda', project_id='casgevy-potency'), catalog_version='approved-alias-catalog-v1-8e2a10ae7742f12f5d9eaaa1eaffa42033183202c27f9b9d6b6c149cce8367fd', latest_jev_catalog_versions=(), latest_extraction_catalog_versions=(), semantic_stale=False, semantic_incomplete=False),
 'counts': {'documents': 24,
  'passages': 791,
  'aliases': 0,
  'alias_evidence': 0,
  'alias_discovery_runs': 0,
  'embedding_runs': 0,
  'indexes': 0,
  'jev_runs': 0,
  'extraction_runs': 0,
  'materializations': 0,
  'graph_refreshes': 0,
  'exports': 1},
 'statuses': {'aliases': {},
  'processing': {'not_run': 791},
  'alias_discovery': {},
  'embedding': {},
  'jev': {},
  'extraction': {},
  'materialization': {},
  'graph_refresh': {},
  'export': {'active': 1}},
 'sanitized_failures': {'alias_discovery': 0,
  'embedding': 0,
  'jev': 0,
  'extraction': 0,
  'materialization': 0,
  '

## Static aliases
Inputs: authoritative seed file. Outputs: seed reconciliation report. Database effect: writes approved static aliases and evidence. Rerun behavior: idempotent. Invalidation: catalog changes require later semantic reruns.

In [6]:
seed_report = load_alias_seeds_stage(config_path)
seed_report


AliasSeedReport(scope=Scope(tenant_id='public-fda', project_id='casgevy-potency'), inserted=3, unchanged=0, retired=0, replaced=0, evidence_inserted=3, catalog_version='approved-alias-catalog-v1-afb7e96cadcdb3a8d1785ba8ce4521381ea69a7d2a8dccc149c6eed3207df3a5')

In [7]:
seed_inspection = inspect_stage(config_path, output_root=output_root)
seed_inspection


{'scope': Scope(tenant_id='public-fda', project_id='casgevy-potency'),
 'catalog': AliasCatalogStatus(scope=Scope(tenant_id='public-fda', project_id='casgevy-potency'), catalog_version='approved-alias-catalog-v1-afb7e96cadcdb3a8d1785ba8ce4521381ea69a7d2a8dccc149c6eed3207df3a5', latest_jev_catalog_versions=(), latest_extraction_catalog_versions=(), semantic_stale=False, semantic_incomplete=False),
 'counts': {'documents': 24,
  'passages': 791,
  'aliases': 3,
  'alias_evidence': 3,
  'alias_discovery_runs': 0,
  'embedding_runs': 0,
  'indexes': 0,
  'jev_runs': 0,
  'extraction_runs': 0,
  'materializations': 0,
  'graph_refreshes': 0,
  'exports': 1},
 'statuses': {'aliases': {'approved': 3},
  'processing': {'not_run': 791},
  'alias_discovery': {},
  'embedding': {},
  'jev': {},
  'extraction': {},
  'materialization': {},
  'graph_refresh': {},
  'export': {'active': 1}},
 'sanitized_failures': {'alias_discovery': 0,
  'embedding': 0,
  'jev': 0,
  'extraction': 0,
  'materializa

## Deterministic discovery
Inputs: active passages. Outputs: pending alias proposals and evidence. Database effect: writes deterministic runs. Rerun behavior: reuses completed runs. Invalidation: new source versions need discovery.

In [8]:
deterministic_report = discover_aliases_deterministic_stage(config_path)
deterministic_report


AliasDiscoveryReport(method='deterministic', documents=24, succeeded=24, failed=0, reused=0, proposals=491, evidence_inserted=491, discovery_run_ids=(UUID('721084a0-b273-460e-bef1-8b2e21dc87c9'), UUID('92bdcc2d-2baa-492f-b648-fb45cb42a1c0'), UUID('aba16488-f362-4684-86a3-c51686aa9a94'), UUID('fe87a3d6-fc82-45fb-851c-185e74bb161d'), UUID('711f74cd-63d7-4c56-a4ae-8074d492bff4'), UUID('1d72763b-3e9e-4f48-aacc-43874b8f678b'), UUID('d1d6132c-470f-4447-af3d-f60c52b8ecf3'), UUID('5670e274-c3a0-4723-9fbf-8799a91bab53'), UUID('df0d8b3e-b114-4e4a-a04a-99cac56f2a22'), UUID('4e1749c1-5432-4dfd-a57b-0030837a6d0e'), UUID('76258437-221c-42d2-9cf3-3f3e0966bfd5'), UUID('f4f8df44-efef-46e9-9dd2-d754b50c2086'), UUID('45885792-3b78-4056-92f0-041d8a31649f'), UUID('944074be-bec4-4ef9-bebd-681021cad313'), UUID('e10fc2cb-144a-4306-bc05-f9a3266d9ded'), UUID('de71f2bc-8cc7-4cdd-a4a5-f4ce219ffcf3'), UUID('85109aa8-c87e-4da5-8f34-320e4de829b8'), UUID('3ddbfbc3-956d-4b88-b045-7f97cff2f0fb'), UUID('235aef18-e51f-4d

In [ ]:
deterministic_inspection = inspect_stage(config_path, output_root=output_root)
deterministic_inspection

See aliases data in http://localhost:8888/lab/tree/notebooks/03_review_aliases.ipynb

## Optional LLM discovery
Inputs: active passages and an explicitly enabled model gate. Outputs: pending proposals. Database effect: writes audited LLM discovery runs. Rerun behavior: reuses matching runs. Invalidation: catalog decisions require semantic reruns.

In [ ]:
llm_discovery_report = discover_aliases_llm_stage(config_path, enabled=enable_llm_alias_discovery)
llm_discovery_report


[alias-llm] discovering aliases in 24 document(s) with claude-opus-5-5
[alias-llm] 1/24 casgevy-125785-analytical-method-review@sha256:aff6f3891ba1da965620d69cb1400a9893f22f33673c820144bfcb1ceda72255: already done, reusing
[alias-llm] 2/24 casgevy-125785-approval-letter@sha256:489de36be50b17defb8c964df772a5c1bbe9be9bd95938d4fb6dfa5c906a2b7a: already done, reusing
[alias-llm] 3/24 casgevy-125785-bioresearch-monitoring-final-review-memo-casgevy@sha256:1edf0e5e0f9da3b57fc87d72cdf727bacb1d0a086cf30e048e8391b8688eedba: already done, reusing
[alias-llm] 4/24 casgevy-125785-cber-bioinformatics-bla-review-memo-casgevy@sha256:8bb2af475a0f7b0def9f98dc39224ca6a3dd0a4201bc9ac0bc480e7816c6e13a: already done, reusing
[alias-llm] 5/24 casgevy-125785-cber-dmpq-cmc-facility-bla-review-memo-casgevy@sha256:6e2ac65c659c16138a0d62fe4f6c4b10345c4d0517bb05d373c4b4bf2759116a: already done, reusing
[alias-llm] 6/24 casgevy-125785-clinical-pharmacology-bla-review-casgevy@sha256:fbedfe2ab305a2fa77d8305d0162674aa

casgevy-125785-pharmacovigilance-plan-review-memo-casgevy@sha256:aa75c6daa83329da4bb2e84a6195bf257cb5f2df9064adc374d60235ab9a0aa7: failed with ValueError, recording failed run 57bf9ed8-b71c-4ac4-8318-e6a77a5ba6e6


[alias-llm] 13/24 casgevy-125785-pharmacovigilance-plan-review-memo-casgevy@sha256:aa75c6daa83329da4bb2e84a6195bf257cb5f2df9064adc374d60235ab9a0aa7: failed (ValueError)
[alias-llm] 14/24 casgevy-125785-statistical-review-casgevy@sha256:b2567388098246119fcd2e75df8da06d6b234a1c7134ef0ccd68df41fa57e021: processing
[alias-llm] 14/24 casgevy-125785-statistical-review-casgevy@sha256:b2567388098246119fcd2e75df8da06d6b234a1c7134ef0ccd68df41fa57e021: done, 6 proposal(s)
[alias-llm] 15/24 casgevy-125787-scd-approval-letter@2023-12-08:bff184adda8256c4: processing
[alias-llm] 15/24 casgevy-125787-scd-approval-letter@2023-12-08:bff184adda8256c4: done, 2 proposal(s)
[alias-llm] 16/24 cfr-21-600-3@2026-09-17:5d8fda0c8ab9d414: processing
[alias-llm] 16/24 cfr-21-600-3@2026-09-17:5d8fda0c8ab9d414: done, 0 proposal(s)
[alias-llm] 17/24 cfr-21-610-10@2026-09-17:4dd03969381e4e80: processing
[alias-llm] 17/24 cfr-21-610-10@2026-09-17:4dd03969381e4e80: done, 0 proposal(s)
[alias-llm] 18/24 guidance-cgt-pote

In [11]:
llm_discovery_inspection = inspect_stage(config_path, output_root=output_root)
llm_discovery_inspection


{'scope': Scope(tenant_id='public-fda', project_id='casgevy-potency'),
 'catalog': AliasCatalogStatus(scope=Scope(tenant_id='public-fda', project_id='casgevy-potency'), catalog_version='approved-alias-catalog-v1-afb7e96cadcdb3a8d1785ba8ce4521381ea69a7d2a8dccc149c6eed3207df3a5', latest_jev_catalog_versions=(), latest_extraction_catalog_versions=(), semantic_stale=False, semantic_incomplete=False),
 'counts': {'documents': 24,
  'passages': 791,
  'aliases': 333,
  'alias_evidence': 494,
  'alias_discovery_runs': 24,
  'embedding_runs': 0,
  'indexes': 0,
  'jev_runs': 0,
  'extraction_runs': 0,
  'materializations': 0,
  'graph_refreshes': 0,
  'exports': 1},
 'statuses': {'aliases': {'pending': 330, 'approved': 3},
  'processing': {'not_run': 791},
  'alias_discovery': {'succeeded': 24},
  'embedding': {},
  'jev': {},
  'extraction': {},
  'materialization': {},
  'graph_refresh': {},
  'export': {'active': 1}},
 'sanitized_failures': {'alias_discovery': 0,
  'embedding': 0,
  'jev': 

## Alias inspection and review
Inputs: pending aliases and evidence. Outputs: read-only review queue. Database effect: none here; review is explicit. Rerun behavior: safe. Invalidation: approval requires an explicit later semantic rebuild.

Use `python -m potency_kg.cli review-alias --config configs/casgevy-potency.json --alias-id ID --decision approved --reviewer NAME --entity-type Product --canonical-key KEY --canonical-name NAME` after inspecting evidence.

In [ ]:
review_queue = inspect_aliases_stage(config_path, status='pending')
selected_alias_id = review_queue[0].alias_id if review_queue else None
review_evidence = inspect_alias_evidence_stage(config_path, alias_id=selected_alias_id) if selected_alias_id else ()
{'aliases': review_queue, 'selected_alias_id': selected_alias_id, 'evidence': review_evidence}


In [ ]:
review_inspection = inspect_stage(config_path, output_root=output_root)
review_inspection


## Embeddings
Inputs: active passages and explicit embedding settings. Outputs: embedding report. Database effect: writes vectors. Rerun behavior: reuses matching runs. Invalidation: source changes require embeddings again.

In [ ]:
embedding_report = embed_stage(config_path)
embedding_report


In [ ]:
embedding_inspection = inspect_stage(config_path, output_root=output_root)
embedding_inspection


## Jev
Inputs: embeddings and explicit Jev settings. Outputs: selection report. Database effect: writes classifications. Rerun behavior: reuses matching responses. Invalidation: alias catalog changes require Jev again.

In [ ]:
jev_report = classify_jev_stage(config_path)
jev_report


In [ ]:
jev_inspection = inspect_stage(config_path, output_root=output_root)
jev_inspection


## Extraction
Inputs: selected passages and explicit generation settings. Outputs: structured extraction report. Database effect: writes validated runs. Rerun behavior: reuses exact runs. Invalidation: alias catalog changes require extraction again.

In [ ]:
extraction_report = extract_stage(config_path)
extraction_report


In [ ]:
extraction_inspection = inspect_stage(config_path, output_root=output_root)
extraction_inspection


## Materialization
Inputs: validated extraction runs. Outputs: materialization report. Database effect: writes relational graph facts. Rerun behavior: idempotent. Invalidation: new semantic runs require materialization.

In [ ]:
materialization_report = materialize_stage(config_path)
materialization_report


In [ ]:
materialization_inspection = inspect_stage(config_path, output_root=output_root)
materialization_inspection


## Refresh
Inputs: materialized relational state. Outputs: activated graph report. Database effect: writes a shadow graph. Rerun behavior: creates or reuses deterministic state. Invalidation: semantic inputs require refresh.

In [ ]:
refresh_report = refresh_graph_stage(config_path, mode='semantic')
refresh_report


In [ ]:
refresh_inspection = inspect_stage(config_path, output_root=output_root)
refresh_inspection


## Export
Inputs: active graph. Outputs: immutable export report. Database effect: records publication state. Rerun behavior: validates and reuses an identical export. Invalidation: a new graph needs export.

In [ ]:
export_report = export_stage(config_path, output_root)
export_report


In [ ]:
export_inspection = inspect_stage(config_path, output_root=output_root)
export_inspection


## Inspection
Inputs: current scope. Outputs: catalog status. Database effect: none. Rerun behavior: safe and read-only. Invalidation: reports current catalog and semantic staleness.

In [ ]:
inspection_report = inspect_stage(config_path, output_root=output_root)
inspection_report
